# Llama 3.3 70B — RAG Generation

Corre **directamente no kernel** (como o `fine_tune.ipynb`) para manter o servidor activo.

Ordem: **Célula 1 → 2 → 3 → 4**

In [ ]:
import os, sys
from pathlib import Path

# ─── Ajusta aqui ──────────────────────────────────────
GPUS         = "1,6"
RAG_CONFIG   = "configs/rag_generation_llama70b_2gpu.yaml"
NORAG_CONFIG = "configs/rag_generation_llama70b_norag_2gpu.yaml"
# ──────────────────────────────────────────────────────

# Deve ser definido antes de importar torch
os.environ["CUDA_VISIBLE_DEVICES"] = GPUS
os.environ["HF_HOME"] = "/home/jovyan/privado/.cache/huggingface"

# Adiciona o projecto ao path para o import funcionar
PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path("/home/jovyan/privado/LLMDermato/derm-rag")
# No notebook, __file__ não existe — usamos o path conhecido:
PROJECT_ROOT = Path("/home/jovyan/privado/LLMDermato/derm-rag")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"CUDA_VISIBLE_DEVICES : {os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"HF_HOME              : {os.environ['HF_HOME']}")
print(f"Python               : {sys.executable}")

In [ ]:
import yaml

ROOT = "/home/jovyan/privado/LLMDermato/derm-rag"

rag_cfg = {
    "data": {
        "derm_csv":  f"{ROOT}/data/processed/derm_cases.csv",
        "id_col":    "patient_uid",
        "text_col":  "patient",
        "title_col": "title",
    },
    "split": {"n_queries": 1000, "seed": 42},
    "retrieval": {
        "rankings_prefix": "crag_v2_reranker_finetuned",
        "results_dir":     f"{ROOT}/results",
        "top_k":           3,
    },
    "presentation": {"max_tokens": 400},
    "generation": {
        "model_name":              "meta-llama/Llama-3.3-70B-Instruct",
        "torch_dtype":             "bfloat16",
        "device_map":              "auto",
        "max_model_len":           8192,
        "max_new_tokens":          512,
        "temperature":             0.1,
        "batch_size":              2,
        "context_case_max_tokens": 600,
    },
    "evaluation": {"results_dir": f"{ROOT}/results", "run_name": "rag_gen_llama70b"},
}

norag_cfg = {
    **rag_cfg,
    "retrieval": {"rankings_prefix": "", "results_dir": f"{ROOT}/results", "top_k": 0},
    "evaluation": {"results_dir": f"{ROOT}/results", "run_name": "rag_gen_llama70b_norag"},
}

for path, cfg in [(f"{ROOT}/{RAG_CONFIG}", rag_cfg), (f"{ROOT}/{NORAG_CONFIG}", norag_cfg)]:
    with open(path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
    print(f"Criado: {path}")

# Actualiza RAG_CONFIG e NORAG_CONFIG para paths absolutos
RAG_CONFIG   = f"{ROOT}/configs/rag_generation_llama70b_2gpu.yaml"
NORAG_CONFIG = f"{ROOT}/configs/rag_generation_llama70b_norag_2gpu.yaml"
print("OK — configs com paths absolutos")

In [ ]:
import getpass
from huggingface_hub import login, whoami

try:
    info = whoami()
    print(f"Já autenticado como: {info['name']}")
except Exception:
    token = getpass.getpass("HuggingFace token (input oculto): ")
    login(token=token)
    print(f"Autenticado como: {whoami()['name']}")

In [ ]:
# ── RAG Generation ──────────────────────────────────────
# Corre no kernel (mantém o servidor activo, igual ao trainer.train())
from scripts.run_rag_generation import main

sys.argv = ["run_rag_generation.py", "--config", RAG_CONFIG]
main()

In [ ]:
# ── No-RAG Generation ───────────────────────────────────
import importlib
import scripts.run_rag_generation as _mod
importlib.reload(_mod)

sys.argv = ["run_rag_generation.py", "--config", NORAG_CONFIG]
_mod.main()

In [ ]:
# Ver resultados gerados
!ls -lh results/ | grep llama70b

---
## RAGAS Evaluation
Corre após ter os resultados do RAG e No-RAG.

In [ ]:
import getpass, importlib, sys, os
from pathlib import Path

ROOT = "/home/jovyan/privado/LLMDermato/derm-rag"

ANTHROPIC_KEY = getpass.getpass("Anthropic API key: ")
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_KEY

# Os 4 ficheiros a avaliar
RESULT_FILES = [
    f"{ROOT}/results/rag_gen_qwen3_8b_20260616_201755.json",
    f"{ROOT}/results/rag_gen_norag_qwen3_8b_20260617_144526.json",
    f"{ROOT}/results/rag_gen_llama70b_20260623_000514.json",
    f"{ROOT}/results/rag_gen_llama70b_norag_20260623_055042.json",
]

for f in RESULT_FILES:
    exists = Path(f).exists()
    print(f"{'✓' if exists else '✗'} {Path(f).name}")


In [ ]:
import importlib
import scripts.evaluate_ragas as _ragas

DATA_CSV = f"{ROOT}/data/processed/derm_cases.csv"

# Avalia os 4 ficheiros sequencialmente
for result_file in RESULT_FILES:
    print(f"\n{'='*60}")
    print(f"Avaliando: {Path(result_file).name}")
    print('='*60)

    importlib.reload(_ragas)
    sys.argv = [
        "evaluate_ragas.py",
        "--results",    result_file,
        "--data",       DATA_CSV,
        "--n-samples",  "200",
        "--model",      "claude-haiku-4-5",
        "--seed",       "42",
    ]
    _ragas.main()
